# **Methods Training - Những phương pháp huấn luyện mô hình**

Do bài toán dự đoán doanh thu đang có nhiều biến động và nhiễu, khiến cho việc huấn luyện các mô hình với các phương pháo thông thường đều rất khó có được kết quả tốt. Vì vậy, trong notebooks này chúng tôi sẽ đưa ra một vài phương pháp chính của chúng tôi cũng như thử nghiệm chúng để cải thiện được cho các mô hình.

Cụ thể:
- Đưa ra và kiểm thử những quy luật chung
- Liệt kê một vài phương pháp khả thi và thử nghiệm chúng
- So sánh và kết luận cho hướng đi tiếp theo

**Mục tiêu**: Tìm ra được phương pháp huấn luyện mô hình tốt nhất

**Mục lục**:

1. 
2. 
3. 
4. 

**Thiết lập và cài đặt**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
import warnings

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath('..'))
from src.get_data import get_data_processed, get_connection
from src.save_data import save_to_processed

In [2]:
conn = get_connection()

[OKE] Kết nối thành công tới database tại C:\Users\YOGA\Desktop\MyProjects\datathon\github\vimchanhxa-datathon\data\database\datathon.duckdb


In [3]:
sales = conn.execute('SELECT * FROM sales').df()
sales.tail()

,date,revenue,cogs
3828,2022-12-27,2100553.66,2184872.24
3829,2022-12-28,3448729.20,3513621.00
3830,2022-12-29,3083944.33,3170787.10
3831,2022-12-30,2884668.76,3022292.15
3832,2022-12-31,2383037.48,2279288.13


In [4]:
pred = conn.execute('SELECT * FROM sales_test').df()
pred.head()

,date,revenue,cogs
0,2023-01-01,2665507.20,2518885.15
1,2023-01-02,1280007.89,1136463.00
2,2023-01-03,1015899.51,822721.12
3,2023-01-04,1142997.27,914554.18
4,2023-01-05,1236312.34,984390.24


In [5]:
print(f"Train Period: {sales['date'].min().date()} to {sales['date'].max().date()}")
print(f"Test Period : {pred['date'].min().date()} to {pred['date'].max().date()}")

Train Period: 2012-07-04 to 2022-12-31
Test Period : 2023-01-01 to 2024-07-01


## 1. Các phương pháp tối ưu

### 1.1 Dữ đoán trực tiếp
ở phần này, chúng ta sẽ khai thác toàn bô những đặc trưng mà chúng ta có thể biết trước được trong tương lai có thể xảy ra. Hướng đi này là một phương pháp dễ tiếp cận và dễ khai thác đặc tính biến động của từng doanh thu qua từng mốc thời gian.

In [6]:
train = sales.copy()
train["is_test"] = 0
test_df = pred[["date"]].copy()
test_df["is_test"] = 1
df = pd.concat([train, test_df], ignore_index=True)
df.tail()

,date,revenue,cogs,is_test
4376,2024-06-27,NaN,NaN,1
4377,2024-06-28,NaN,NaN,1
4378,2024-06-29,NaN,NaN,1
4379,2024-06-30,NaN,NaN,1
4380,2024-07-01,NaN,NaN,1


Tạo các features dài hạn theo thời gian

In [7]:
df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_month"] = df["date"].dt.day
df["day_of_year"] = df["date"].dt.dayofyear
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year
df["is_weekend"] = np.where(df["day_of_week"] >= 5, 1, 0)
df["is_payday"] = np.where((df["day_of_month"] >= 25) | (df["day_of_month"] <= 5), 1, 0)
df["is_double_day"] = np.where(df["month"] == df["day_of_month"], 1, 0)
df["days_since_start"] = (df["date"] - df["date"].min()).dt.days

Features những ngày hội, đặc biệt là ngày têt khi eda đã cho thấy rằng khoảng thời gian tết, doanh thu đã có đỉnh chóp khá cao

In [8]:
tet_dates = {
    2012: "2012-01-23",
    2013: "2013-02-10",
    2014: "2014-01-31",
    2015: "2015-02-19",
    2016: "2016-02-08",
    2017: "2017-01-28",
    2018: "2018-02-16",
    2019: "2019-02-05",
    2020: "2020-01-25",
    2021: "2021-02-12",
    2022: "2022-02-01",
    2023: "2023-01-22",
    2024: "2024-02-10",
}
tet_map = pd.DataFrame(list(tet_dates.items()), columns=["year", "tet_date"])
tet_map["tet_date"] = pd.to_datetime(tet_map["tet_date"])

df = df.merge(tet_map, on="year", how="left")
df["days_to_tet"] = (df["tet_date"] - df["date"]).dt.days
df["is_pre_tet_rush"] = np.where((df["days_to_tet"] > 0) & (df["days_to_tet"] <= 21), 1, 0)
df["is_tet_holiday"] = np.where((df["days_to_tet"] <= 0) & (df["days_to_tet"] >= -6), 1, 0)
df.drop(columns=["tet_date"], inplace=True)

In [9]:
df.head()

,date,revenue,cogs,is_test,day_of_week,day_of_month,day_of_year,month,year,is_weekend,is_payday,is_double_day,days_since_start,days_to_tet,is_pre_tet_rush,is_tet_holiday
0,2012-07-04,5123547.94,3982991.19,0,2,4,186,7,2012,0,1,0,0,-163,0,0
1,2012-07-05,2751773.45,2150580.23,0,3,5,187,7,2012,0,1,0,1,-164,0,0
2,2012-07-06,3054029.42,2517632.84,0,4,6,188,7,2012,0,0,0,2,-165,0,0
3,2012-07-07,2667930.94,2108246.62,0,5,7,189,7,2012,1,0,1,3,-166,0,0
4,2012-07-08,2360851.90,1808622.79,0,6,8,190,7,2012,1,0,0,4,-167,0,0


Như ở phần `EDA`, chúng ta đã thấy rằng doanh nghiệp hiện tại đang mở các đợt giảm giá theo chu kỳ lặp lại hằng năm và hai năm. Cụ thể là với các mã giảm giá `Spring Sale`, `Mid-Year Sale`, `Fall Launch`, `Year-End Sale` đang được mở lại qua từng năm, còn với `Urban Blowout` và `Rural Special` thì lại lặp 2 năm một lần ( các năm lẻ ). Đặc biệt hơn là thời gian bắt đầu và kết thúc cũng rất khớp nhau, cùng lắm chỉ lệch 1 ngày. Tuy nhiên có một vài biến trong đó sẽ không được lặp lại giống hệt các năm trước như `min_order_value`, vì vậy dưới đây ta sẽ làm lại dữ liệu promotions này.

In [10]:
promotions = conn.execute('SELECT * FROM promotions').df()
promotions.head()

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,PROMO-0001,Spring Sale 2013,percentage,12.0,2013-03-18,2013-04-17,None,email,1,0.0
1,PROMO-0002,Mid-Year Sale 2013,percentage,18.0,2013-06-23,2013-07-22,None,online,0,0.0
2,PROMO-0003,Fall Launch 2013,percentage,10.0,2013-08-30,2013-10-02,None,email,0,0.0
3,PROMO-0004,Year-End Sale 2013,percentage,20.0,2013-11-18,2014-01-02,None,all_channels,0,50000.0
4,PROMO-0005,Urban Blowout 2013,fixed,50.0,2013-07-30,2013-09-02,Streetwear,online,0,150000.0


Do vẫn còn sự khác nhau, nên chúng tôi sẽ tạo thêm chỉ một cột đặc trưng để tránh bị loãng mô hình bao gồm: `is_promo` và `total_discount`, những ngày giảm giá và phần trăm đang giảm giá. Mục đích là hạn chế tạo quá nhiều cột nhiễu không cần thiết và vẫn thể hiện được sự khác biệt so với các mã giảm giá và các ngày khác. Sau đó, nếu `total_discount` không thể hiện được rõ vai trò, chúng tôi sẽ xóa đi vì trong dữ liệu sẽ có một vài mã có `stackable_flag` là không cho sử dụng 2 mã cùng lúc.

In [11]:
# Lọc các mã giảm giá lặp lại
promo_range = pd.date_range(start=df["date"].min(), end=df["date"].max())

daily_p = pd.DataFrame({
    "date": promo_range,
    "is_promo": 0,
    "total_discount": 0.0
})

for _, row in promotions.iterrows():
    mask = (
        (daily_p["date"] >= row["start_date"]) &
        (daily_p["date"] <= row["end_date"])
    )
    daily_p.loc[mask, "is_promo"] = 1
    daily_p.loc[mask, "total_discount"] += row["discount_value"]

df = (df.merge(daily_p, on="date", how="left").fillna({"is_promo": 0, "total_discount": 0}))

In [12]:
df.is_promo.value_counts()

is_promo
0    2674
1    1707
Name: count, dtype: int64

Dữ liệu ở trên được coi như là toàn bộ dữ liệu biết trước tương lai. Do đó đây sẽ là phương pháp đầu tiên mà chúng tôi sẽ tiếp cận và dễ nhất trước khi đi vào các phương pháp khác phức tạp hơn. Ngoài ra đây sẽ là một baseline mới để có thể xác địnhh được thói quen của khách hàng qua từng khoảng thời gian, từ đó các features tạo ra từ phương pháp này cũng sẽ được sử dụng vào các phương pháp khác.

### 1.2 Dự đoán gián tiếp
Trong phần này, chúng tôi sẽ kiểm tra toàn bộ đặc trưng của tập dữ liệu có mật thiết cao tới `revenue`. Cụ thể, chúng tôi sẽ xác định những yếu tố nào đang làm ảnh hưởng tới `revenue` cao nhất. Từ đó, chúng tôi sẽ xây dựng mô hình để dự đoán những yếu tố đó rồi sử dụng dữ liệu dự đoán để tính ra kết quả cuối cùng. Hiện tại sau khi khám phá thì có những đặc trưng sau có mức tương quan cực cao tới `revenue`:
- `unique_customer`: Tổng số khách hàng trong một ngày
- `total_order`: Tổng số đơn hàng trong một ngày
- `total_product`: Tổng số sản phẩm bán được trong một ngày

In [13]:
orders = conn.execute('SELECT * FROM orders').df()
orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


In [14]:
order_items = conn.execute('SELECT * FROM order_items').df()
order_items.head()

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,None,None
1,2,609,7,10166.25,0.0,None,None
2,3,396,3,11220.33,0.0,None,None
3,4,635,5,10639.25,0.0,None,None
4,6,1935,1,1597.84,0.0,None,None


In [15]:
# 2. Merge hai bảng dựa trên order_id
# Chúng ta dùng 'inner' để đảm bảo chỉ lấy những đơn hàng có chi tiết sản phẩm
df_merged = pd.merge(orders, order_items, on='order_id', how='inner')

# 4. Tổng hợp dữ liệu theo Ngày (Daily Aggregation)
# Đây là bước quan trọng để đưa dữ liệu về dạng Time-series
df_daily_revenue = df_merged.groupby('order_date').agg(
    order_count=('order_id', 'nunique'),        # Số lượng đơn hàng trong ngày
    unique_customers=('customer_id', 'nunique'), # Số lượng khách hàng trong ngày
    total_quantity=('quantity', 'sum')          # Tổng số lượng sản phẩm bán ra
).reset_index()

# 5. Sắp xếp theo thứ tự thời gian
df_daily_revenue = df_daily_revenue.sort_values('order_date')

print("Dữ liệu sau khi merge và tổng hợp theo ngày:")
df_daily_revenue.head()

Dữ liệu sau khi merge và tổng hợp theo ngày:


,order_date,order_count,unique_customers,total_quantity
0,2012-07-04,162,161,777
1,2012-07-05,97,97,428
2,2012-07-06,93,93,441
3,2012-07-07,73,73,364
4,2012-07-08,88,87,394


In [16]:
for col in ["order_count", "unique_customers", "total_quantity"]:
    print(f"Correlations {col} with Revenue: {df_daily_revenue[col].corr(df['revenue']):.2f}")

Correlations order_count with Revenue: 0.94
Correlations unique_customers with Revenue: 0.94
Correlations total_quantity with Revenue: 0.92


Dựa vào đó, chúng tôi sẽ xây dựng các mô hình dự đoán riêng biệt, rồi sau đó tập hợp lại để dự đoán cho `revenue`

### 1.4 Dựa đoán chi tiết
Ở phần này, sau khi chúng tôi nhận thấy rằng nếu dự đoán gián tiếp `revenue` thì có thể dự đoán dựa trên nhu cầu sản phầm, tức dự đoán từng sản phầm sẽ được mua trong năm tới. Lý do bởi vì không chỉ có rất nhiều thông tin mà còn cực kỳ chi tiết. Tuy nhiên cũng giống như bước trên, mô hình có thể sẽ dự đoán sai và khi tính `revenue`, sai số sẽ lại bị cộng dồn.

In [17]:
products = conn.execute('SELECT * FROM products').df()
products.head()

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650,9704.843
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076,5393.870
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633,11371.919
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717,8573.173
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.335,14063.570


## 2. Trích xuất dữ liệu
Dựa trên 3 phương pháp trên, trong phần này chúng tôi sẽ xây dựng toàn bộ tập dữ liệu để chuẩn bị cho 3 phương pháp này. Đây chỉ mới là tập hợp dữ liệu. Sau đó chúng tôi vẫn có thể thêm các đặc trưng hữu ích cho từng mô hình.

### 2.1 Dự đoán trực tiếp

In [18]:
# Lưu dữ liệu đã xử lý vào thư mục processed
save_to_processed(df, "methods_1.csv")

Đã lưu thành công tại: C:\Users\YOGA\Desktop\MyProjects\datathon\github\vimchanhxa-datathon\data\processed\methods_1.csv


### 2.2 Dự đoán gián tiếp

---
**Kết luận:**

---
Notebooks tiếp theo: [04_FEATURES_SELECTION_.ipynb](04_FEATURES_SELECTION_.ipynb) - Xây dựng các đặc trưng cho mô hình